In [8]:
import os
import re
import cv2
import time
import math
from  glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

!pip -q install ultralytics

from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict

import xml.etree.ElementTree as ET
import shutil

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


# CONFIGURATION

In [10]:
DATASET_DIR = Path("/kaggle/working/data")

IMG_VAL_DIR = DATASET_DIR / "images" / "val"
LBL_VAL_DIR = DATASET_DIR / "labels" / "val"

MODEL_PATH = "/kaggle/input/models/chaymaeramii/modelname/other/default/1/best.pt"

# Car-only model
NUM_CLASSES = 1
CLASS_NAMES = {0: "car"}

IMG_SIZE = 768
CONF_THRES = 0.30     
IOU_NMS = 0.60

# Event / adaptive
K_MIN = 1
K_MAX = 5
N_MAX_CARS = 20

# Pour ne pas overfitter les seuils sur toute la validation
CALIBRATION_SEQUENCE_RATIO = 0.20

RESULTS_DIR = Path("/kaggle/working/ua_detrac_event_adaptive_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Images val:", IMG_VAL_DIR)
print("Labels val:", LBL_VAL_DIR)
print("Model:", MODEL_PATH)
print("Results:", RESULTS_DIR)

Images val: /kaggle/working/data/images/val
Labels val: /kaggle/working/data/labels/val
Model: /kaggle/input/models/chaymaeramii/modelname/other/default/1/best.pt
Results: /kaggle/working/ua_detrac_event_adaptive_results


In [11]:
def list_images_recursive(img_dir):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp"]
    files = []
    for ext in exts:
        files.extend(img_dir.rglob(ext))
    return sorted(files)


def image_id(path):
    return Path(path).stem


def extract_frame_number(path):
    stem = Path(path).stem
    nums = re.findall(r"\d+", stem)
    if len(nums) == 0:
        return 0
    return int(nums[-1])


def infer_sequence_id(path, root_dir):
    """
    UA-DETRAC est normalement organisé par séquences MVI_xxxx.
    Si les images sont dans des dossiers, le parent devient la séquence.
    Sinon, on essaie d'extraire MVI_xxxx du nom.
    """
    path = Path(path)
    rel = path.relative_to(root_dir)

    if len(rel.parts) > 1:
        return rel.parts[0]

    stem = path.stem

    m = re.search(r"(MVI[_-]?\d+)", stem, re.IGNORECASE)
    if m:
        return m.group(1)

    # fallback : préfixe avant le dernier numéro
    seq = re.sub(r"[_-]?\d+$", "", stem)
    if seq == "":
        seq = "single_stream"

    return seq


def build_sequences(img_dir):
    images = list_images_recursive(img_dir)
    sequences = defaultdict(list)

    for p in images:
        sid = infer_sequence_id(p, img_dir)
        sequences[sid].append(p)

    ordered_sequences = {}
    for sid, paths in sequences.items():
        ordered_sequences[sid] = sorted(paths, key=extract_frame_number)

    return ordered_sequences


sequences = build_sequences(IMG_VAL_DIR)

print("Nombre de séquences:", len(sequences))
print("Nombre total images:", sum(len(v) for v in sequences.values()))

for sid, paths in list(sequences.items())[:5]:
    print(sid, ":", len(paths), "frames", "ex:", paths[0].name)

Nombre de séquences: 40
Nombre total images: 56167
MVI_39031 : 1470 frames ex: MVI_39031_img00001.jpg
MVI_39051 : 1051 frames ex: MVI_39051_img00001.jpg
MVI_39211 : 1566 frames ex: MVI_39211_img00001.jpg
MVI_39271 : 1566 frames ex: MVI_39271_img00001.jpg
MVI_39311 : 1505 frames ex: MVI_39311_img00001.jpg


# Calibration

In [12]:
def load_image_shape(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f"Image non lisible: {img_path}")
    h, w = img.shape[:2]
    return h, w



def find_label_path(img_path, img_root, label_root):
    """
    Cherche le .txt correspondant à une image.
    Supporte :
    - labels/val/sequence/frame.txt
    - labels/val/frame.txt
    """
    img_path = Path(img_path)
    rel = img_path.relative_to(img_root)

    candidate1 = label_root / rel.with_suffix(".txt")
    if candidate1.exists():
        return candidate1

    candidate2 = label_root / f"{img_path.stem}.txt"
    if candidate2.exists():
        return candidate2

    return candidate1



def yolo_label_to_xyxy(label_path, img_w, img_h):
    """
    YOLO label:
    cls xc yc w h
    Retour:
    [cls, x1, y1, x2, y2]
    """
    boxes = []

    if not Path(label_path).exists():
        return np.zeros((0, 5), dtype=np.float32)

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        cls = int(float(parts[0]))
        xc, yc, bw, bh = map(float, parts[1:5])

        x1 = (xc - bw / 2) * img_w
        y1 = (yc - bh / 2) * img_h
        x2 = (xc + bw / 2) * img_w
        y2 = (yc + bh / 2) * img_h

        x1 = max(0, min(img_w - 1, x1))
        y1 = max(0, min(img_h - 1, y1))
        x2 = max(0, min(img_w - 1, x2))
        y2 = max(0, min(img_h - 1, y2))

        boxes.append([cls, x1, y1, x2, y2])

    return np.array(boxes, dtype=np.float32)


In [13]:
sequence_ids = sorted(list(sequences.keys()))

n_calib = max(1, int(len(sequence_ids) * CALIBRATION_SEQUENCE_RATIO))

calib_seq_ids = sequence_ids[:n_calib]
eval_seq_ids = sequence_ids[n_calib:]

if len(eval_seq_ids) == 0:
    eval_seq_ids = calib_seq_ids

print("Calibration sequences:", len(calib_seq_ids))
print("Evaluation sequences:", len(eval_seq_ids))

print("Calib examples:", calib_seq_ids[:5])
print("Eval examples:", eval_seq_ids[:5])

Calibration sequences: 8
Evaluation sequences: 32
Calib examples: ['MVI_39031', 'MVI_39051', 'MVI_39211', 'MVI_39271', 'MVI_39311']
Eval examples: ['MVI_39501', 'MVI_39511', 'MVI_40701', 'MVI_40711', 'MVI_40712']


In [14]:
def flatten_sequences(seq_ids):
    out = []
    for sid in seq_ids:
        for p in sequences[sid]:
            out.append((sid, p))
    return out


calib_stream = flatten_sequences(calib_seq_ids)
eval_stream = flatten_sequences(eval_seq_ids)

print("Calibration frames:", len(calib_stream))
print("Evaluation frames:", len(eval_stream))

Calibration frames: 11963
Evaluation frames: 44204


In [ ]:
all_stream = flatten_sequences(sequence_ids)

Computing visual deltas: 100%|██████████| 56167/56167 [05:07<00:00, 182.57it/s]


In [16]:
MODEL_PATH = "/kaggle/input/models/chaymaerami/modelyolo/other/default/1/best (4).pt"
model = YOLO(MODEL_PATH)

In [17]:
def run_yolo_single(model, img_path):
    """
    Retourne:
    preds = [x1, y1, x2, y2, conf, cls]
    latency en secondes
    """
    start = time.perf_counter()

    results = model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=CONF_THRES,
        iou=IOU_NMS,
        verbose=False
    )

    latency = time.perf_counter() - start
    r = results[0]

    if r.boxes is None or len(r.boxes) == 0:
        return np.zeros((0, 6), dtype=np.float32), latency

    xyxy = r.boxes.xyxy.cpu().numpy()
    conf = r.boxes.conf.cpu().numpy().reshape(-1, 1)
    cls = r.boxes.cls.cpu().numpy().reshape(-1, 1)

    preds = np.concatenate([xyxy, conf, cls], axis=1).astype(np.float32)
    return preds, latency


def make_key(sid, img_path):
    return f"{sid}/{Path(img_path).stem}"


def build_cache(stream):
    pred_cache = {}
    gt_cache = {}
    latency_cache = {}
    shape_cache = {}

    for sid, img_path in tqdm(stream, desc="YOLO cache"):
        key = make_key(sid, img_path)

        h, w = load_image_shape(img_path)
        label_path = find_label_path(img_path, IMG_VAL_DIR, LBL_VAL_DIR)

        gt = yolo_label_to_xyxy(label_path, w, h)
        preds, latency = run_yolo_single(model, img_path)

        pred_cache[key] = preds
        gt_cache[key] = gt
        latency_cache[key] = latency
        shape_cache[key] = (h, w)

    return pred_cache, gt_cache, latency_cache, shape_cache


pred_cache, gt_cache, latency_cache, shape_cache = build_cache(all_stream)

print("Mean YOLO latency:", np.mean(list(latency_cache.values())) * 1000, "ms")
print("Median YOLO latency:", np.median(list(latency_cache.values())) * 1000, "ms")

YOLO cache: 100%|██████████| 56167/56167 [19:29<00:00, 48.01it/s]

Mean YOLO latency: 17.603939254917915 ms
Median YOLO latency: 17.09589500023867 ms


# Evaluations Functions

In [ ]:
def box_iou_np(box, boxes):
    if boxes.shape[0] == 0:
        return np.zeros((0,), dtype=np.float32)

    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])

    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)

    area_box = np.maximum(0, box[2] - box[0]) * np.maximum(0, box[3] - box[1])
    area_boxes = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])

    union = area_box + area_boxes - inter + 1e-9
    return inter / union


def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))

    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])

    x = np.linspace(0, 1, 101)
    ap = np.trapz(np.interp(x, mrec, mpre), x)
    return float(ap)


def evaluate_class_ap(preds_by_key, gt_by_key, cls_id=0, iou_thr=0.5):
    gt_for_class = {}
    n_gt = 0

    for key, gt in gt_by_key.items():
        gtc = gt[gt[:, 0] == cls_id][:, 1:5] if len(gt) else np.zeros((0, 4))
        gt_for_class[key] = {
            "boxes": gtc,
            "matched": np.zeros((gtc.shape[0],), dtype=bool)
        }
        n_gt += gtc.shape[0]

    preds = []
    for key, pred in preds_by_key.items():
        if pred.shape[0] == 0:
            continue

        pc = pred[pred[:, 5] == cls_id]
        for row in pc:
            preds.append((key, float(row[4]), row[:4]))

    if n_gt == 0:
        return None

    if len(preds) == 0:
        return 0.0

    preds = sorted(preds, key=lambda x: x[1], reverse=True)

    tp = np.zeros(len(preds))
    fp = np.zeros(len(preds))

    for i, (key, conf, box) in enumerate(preds):
        gt_boxes = gt_for_class[key]["boxes"]
        matched = gt_for_class[key]["matched"]

        if gt_boxes.shape[0] == 0:
            fp[i] = 1
            continue

        ious = box_iou_np(box, gt_boxes)
        best_idx = int(np.argmax(ious))
        best_iou = ious[best_idx]

        if best_iou >= iou_thr and not matched[best_idx]:
            tp[i] = 1
            matched[best_idx] = True
        else:
            fp[i] = 1

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)

    recall = tp_cum / (n_gt + 1e-9)
    precision = tp_cum / (tp_cum + fp_cum + 1e-9)

    return compute_ap(recall, precision)


def evaluate_map(preds_by_key, gt_by_key):
    ap50 = evaluate_class_ap(preds_by_key, gt_by_key, cls_id=0, iou_thr=0.50)

    iou_thresholds = np.arange(0.50, 0.96, 0.05)
    aps = []

    for thr in iou_thresholds:
        ap = evaluate_class_ap(preds_by_key, gt_by_key, cls_id=0, iou_thr=thr)
        if ap is not None:
            aps.append(ap)

    return {
        "mAP50": float(ap50 if ap50 is not None else 0.0),
        "mAP50_95": float(np.mean(aps) if len(aps) else 0.0)
    }


def compute_prf1(preds_by_key, gt_by_key, iou_thr=0.5):
    total_tp = 0
    total_fp = 0
    total_fn = 0

    for key, gt in gt_by_key.items():
        pred = preds_by_key.get(key, np.zeros((0, 6), dtype=np.float32))

        gt_boxes = gt[gt[:, 0] == 0][:, 1:5] if len(gt) else np.zeros((0, 4))
        pred_boxes = pred[pred[:, 5] == 0][:, :4] if len(pred) else np.zeros((0, 4))

        matched = np.zeros((gt_boxes.shape[0],), dtype=bool)

        if pred_boxes.shape[0] == 0:
            total_fn += gt_boxes.shape[0]
            continue

        for pb in pred_boxes:
            if gt_boxes.shape[0] == 0:
                total_fp += 1
                continue

            ious = box_iou_np(pb, gt_boxes)
            best_idx = int(np.argmax(ious))
            best_iou = ious[best_idx]

            if best_iou >= iou_thr and not matched[best_idx]:
                total_tp += 1
                matched[best_idx] = True
            else:
                total_fp += 1

        total_fn += int(np.sum(~matched))

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall = total_tp / (total_tp + total_fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(total_tp),
        "fp": int(total_fp),
        "fn": int(total_fn)
    }

In [19]:
def count_mae(preds_by_key, gt_by_key):
    errors = []
    for key, gt in gt_by_key.items():
        pred = preds_by_key.get(key, np.zeros((0, 6), dtype=np.float32))
        n_gt = int(np.sum(gt[:, 0] == 0)) if len(gt) else 0
        n_pred = int(np.sum(pred[:, 5] == 0)) if len(pred) else 0
        errors.append(abs(n_pred - n_gt))

    return float(np.mean(errors)) if len(errors) else 0.0


def temporal_count_stability(stream, preds_by_key):
    """
    Stability proche de 1 = nombre de voitures stable entre frames successives.
    """
    values = []

    prev_sid = None
    prev_count = None

    for sid, img_path in stream:
        key = make_key(sid, img_path)
        pred = preds_by_key.get(key, np.zeros((0, 6), dtype=np.float32))
        count = int(np.sum(pred[:, 5] == 0)) if len(pred) else 0

        if prev_sid == sid and prev_count is not None:
            denom = max(count, prev_count, 1)
            stability = 1.0 - abs(count - prev_count) / denom
            values.append(stability)

        prev_sid = sid
        prev_count = count

    return float(np.mean(values)) if len(values) else 0.0

In [ ]:
def subset_caches(stream):
    keys = [make_key(sid, p) for sid, p in stream]

    pred_sub = {k: pred_cache[k] for k in keys}
    gt_sub = {k: gt_cache[k] for k in keys}
    latency_sub = {k: latency_cache[k] for k in keys}

    return pred_sub, gt_sub, latency_sub




In [22]:
def summarize_policy(policy_name, stream, preds_by_key, infer_flags, gt_by_key, latency_by_key, trigger_latency_per_frame=0.001):
    n_total = len(infer_flags)
    n_yolo = int(sum(infer_flags.values()))
    n_skip = n_total - n_yolo

    map_res = evaluate_map(preds_by_key, gt_by_key)
    prf = compute_prf1(preds_by_key, gt_by_key, iou_thr=0.5)

    yolo_time = sum(latency_by_key[k] for k, flag in infer_flags.items() if flag == 1)
    trigger_time = n_total * trigger_latency_per_frame
    total_time = yolo_time + trigger_time

    count_error = count_mae(preds_by_key, gt_by_key)
    stability = temporal_count_stability(stream, preds_by_key)

    return {
        "policy": policy_name,
        "mAP50": map_res["mAP50"],
        "mAP50_95": map_res["mAP50_95"],
        "precision": prf["precision"],
        "recall": prf["recall"],
        "f1": prf["f1"],
        "count_MAE": count_error,
        "temporal_stability": stability,
        "total_frames": n_total,
        "yolo_frames": n_yolo,
        "skipped_frames": n_skip,
        "frames_processed_ratio": n_yolo / max(n_total, 1),
        "frame_reduction": 1.0 - n_yolo / max(n_total, 1),
        "estimated_total_time_s": total_time,
        "estimated_yolo_time_s": yolo_time,
        "mean_latency_per_frame_ms": 1000 * total_time / max(n_total, 1),
        "effective_fps": n_total / max(total_time, 1e-9)
    }

# Stride k=2 --> k=6

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def build_fixed_stride_policy(stream, pred_cache, stride):
    """
    Fixed-stride scheduling.

    K=1:
        D D D D D ...

    K=2:
        D R D R D R ...

    K=3:
        D R R D R R ...

    D = detector execution
    R = direct reuse of last detector output

    The stride counter is reset at the beginning
    of every video sequence.
    """

    preds_by_key = {}
    infer_flags = {}

    prev_sid = None
    sequence_frame_idx = 0
    last_preds = None

    for sid, img_path in stream:

        key = make_key(sid, img_path)

        # --------------------------------------------
        # New video sequence
        # --------------------------------------------
        if sid != prev_sid:
            sequence_frame_idx = 0
            last_preds = None

        # --------------------------------------------
        # Detector every K frames
        # --------------------------------------------
        run_detector = (
            sequence_frame_idx % stride == 0
        )

        if run_detector:

            last_preds = np.array(
                pred_cache[key],
                copy=True
            )

            infer_flags[key] = 1

        else:

            # Direct last-output reuse
            infer_flags[key] = 0

        preds_by_key[key] = np.array(
            last_preds,
            copy=True
        )

        sequence_frame_idx += 1
        prev_sid = sid

    return preds_by_key, infer_flags

In [ ]:
calib_pred_base, calib_gt, calib_latency = subset_caches(calib_stream)
eval_pred_base, eval_gt, eval_latency = subset_caches(eval_stream)

In [53]:
# ============================================================
# FIXED-STRIDE SWEEP
# ============================================================

STRIDES = [ 2, 3, 4, 5, 6, 7, 8]

fixed_stride_results = []

fixed_stride_outputs = {}

for k in STRIDES:

    print(f"\nEvaluating Fixed Stride K={k} ...")

    preds_k, flags_k = build_fixed_stride_policy(
        stream=eval_stream,
        pred_cache=pred_cache,
        stride=k
    )

    # Store outputs if needed later
    fixed_stride_outputs[k] = {
        "preds": preds_k,
        "flags": flags_k
    }

    summary_k = summarize_policy(
        policy_name=f"Fixed Stride K={k}",
        stream=eval_stream,
        preds_by_key=preds_k,
        infer_flags=flags_k,
        gt_by_key=eval_gt,
        latency_by_key=latency_cache,

        # Fixed stride has essentially no event/risk
        # scheduling computation.
        trigger_latency_per_frame=0.0
    )

    summary_k["stride"] = k

    fixed_stride_results.append(
        summary_k
    )

    print(
        f"K={k} | "
        f"Reduction={100*summary_k['frame_reduction']:.2f}% | "
        f"mAP50={100*summary_k['mAP50']:.2f}% | "
    )


Evaluating Fixed Stride K=2 ...


/tmp/ipykernel_58/4064674785.py:27: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  ap = np.trapz(np.interp(x, mrec, mpre), x)


K=2 | Reduction=49.99% | mAP50=84.80% | 

Evaluating Fixed Stride K=3 ...
K=3 | Reduction=66.64% | mAP50=84.70% | 

Evaluating Fixed Stride K=4 ...
K=4 | Reduction=74.97% | mAP50=84.14% | 

Evaluating Fixed Stride K=5 ...
K=5 | Reduction=80.00% | mAP50=82.75% | 

Evaluating Fixed Stride K=6 ...
K=6 | Reduction=83.30% | mAP50=80.65% | 

Evaluating Fixed Stride K=7 ...
K=7 | Reduction=85.68% | mAP50=78.19% | 

Evaluating Fixed Stride K=8 ...
K=8 | Reduction=87.47% | mAP50=75.62% | 


In [54]:
stride_df = pd.DataFrame(
    fixed_stride_results
)

display(
    stride_df[
        [
            "stride",
            "mAP50",
            "recall",
            "f1",
            "precision",
            "count_MAE",
            "yolo_frames",
            "skipped_frames",
            "frame_reduction",
            "mean_latency_per_frame_ms",
            "effective_fps"
        ]
    ]
)

,stride,mAP50,recall,f1,precision,count_MAE,yolo_frames,skipped_frames,frame_reduction,mean_latency_per_frame_ms,effective_fps
0,2,0.847994,0.827685,0.790750,0.756970,3.260723,22108,22096,0.499864,8.732229,114.518303
1,3,0.847003,0.826443,0.789659,0.756010,3.259388,14745,29459,0.666433,5.825425,171.661280
2,4,0.841447,0.821383,0.784657,0.751074,3.265270,11063,33141,0.749729,4.369257,228.871850
3,5,0.827483,0.811765,0.775752,0.742798,3.254253,8842,35362,0.799973,3.494665,286.150428
4,6,0.806527,0.797008,0.761395,0.728828,3.270767,7381,36823,0.833024,2.914709,343.087422
5,7,0.781949,0.780598,0.745809,0.713989,3.274274,6329,37875,0.856823,2.500587,399.906156
6,8,0.756216,0.762576,0.728698,0.697701,3.270360,5538,38666,0.874717,2.189376,456.751187


In [35]:
import numpy as np
import pandas as pd

# ============================================================
# FIXED STRIDE K=3
# ============================================================

K = 3

stride3_preds, stride3_flags = build_fixed_stride_policy(
    stream=eval_stream,
    pred_cache=eval_gt,
    stride=K
)

# ============================================================
# LATENCY STATISTICS
# ============================================================

def compute_latency_statistics(
    infer_flags,
    latency_cache,
    trigger_latency_per_frame=0.0
):
    frame_latencies = []

    for key, flag in infer_flags.items():

        if flag == 1:
            latency = latency_cache[key] + trigger_latency_per_frame
        else:
            latency = trigger_latency_per_frame

        frame_latencies.append(latency * 1000.0)

    frame_latencies = np.asarray(
        frame_latencies,
        dtype=np.float64
    )

    return {
        "mean_ms": float(np.mean(frame_latencies)),
        "median_ms": float(np.median(frame_latencies)),
        "p95_ms": float(np.percentile(frame_latencies, 95)),
        "p99_ms": float(np.percentile(frame_latencies, 99)),
        "min_ms": float(np.min(frame_latencies)),
        "max_ms": float(np.max(frame_latencies)),
    }


# Fixed stride has no adaptive scheduling overhead
lat_stride3 = compute_latency_statistics(
    infer_flags=stride3_flags,
    latency_cache=eval_latency,
    trigger_latency_per_frame=0.0
)

# ============================================================
# DISPLAY
# ============================================================

latency_stride3_df = pd.DataFrame([
    {
        "policy": "Fixed Stride (K=3)",
        **lat_stride3
    }
])


print("\nFixed Stride K=3")
print("-------------------------")
print(f"Mean latency   : {lat_stride3['mean_ms']:.2f} ms")
print(f"P95 latency    : {lat_stride3['p95_ms']:.2f} ms")
print(f"P99 latency    : {lat_stride3['p99_ms']:.2f} ms")


# ============================================================
# SAVE
# ============================================================

latency_stride3_df.to_csv(
    RESULTS_DIR / "latency_stride3.csv",
    index=False
)

print(
    "\nSaved:",
    RESULTS_DIR / "latency_stride3.csv"
)


Fixed Stride K=3
-------------------------
Mean latency   : 5.83 ms
P95 latency    : 18.61 ms
P99 latency    : 19.89 ms

Saved: /kaggle/working/ua_detrac_event_adaptive_results/latency_stride3.csv


## Condition Sequences

In [36]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import defaultdict

# =====================================================
# Path vers les annotations XML
# =====================================================

xml_dir = Path(
    "/kaggle/input/datasets/bratjay/ua-detrac-orig/"
    "DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML"
)

# =====================================================
# Lire la condition météo depuis chaque XML
# =====================================================

def extract_weather_from_xml(xml_path):
    """
    Extract sequence name and weather condition from UA-DETRAC XML.

    UA-DETRAC uses the attribute name 'sence_weather'
    inside <sequence_attribute ... />.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    sequence_name = root.attrib.get("name", xml_path.stem)

    seq_attr = root.find("sequence_attribute")
    if seq_attr is None:
        return sequence_name, "unknown"

    weather = seq_attr.attrib.get("sence_weather", "unknown")
    weather = weather.strip().lower()

    return sequence_name, weather


# =====================================================
# Construire condition_sequences automatiquement
# =====================================================

condition_sequences = defaultdict(list)

for xml_path in sorted(xml_dir.glob("*.xml")):
    seq_name, weather = extract_weather_from_xml(xml_path)
    condition_sequences[weather].append(seq_name)

condition_sequences = dict(condition_sequences)

# Affichage
for condition, seqs in condition_sequences.items():
    print(f"{condition}: {len(seqs)} sequences")
    print(seqs[:10])
    print("-" * 60)

sunny: 8 sequences
['MVI_39031', 'MVI_39051', 'MVI_39211', 'MVI_39271', 'MVI_39311', 'MVI_39361', 'MVI_39371', 'MVI_39401']
------------------------------------------------------------
cloudy: 11 sequences
['MVI_39501', 'MVI_39511', 'MVI_40701', 'MVI_40711', 'MVI_40712', 'MVI_40714', 'MVI_40851', 'MVI_40852', 'MVI_40853', 'MVI_40854']
------------------------------------------------------------
night: 12 sequences
['MVI_40742', 'MVI_40743', 'MVI_40761', 'MVI_40762', 'MVI_40763', 'MVI_40771', 'MVI_40772', 'MVI_40773', 'MVI_40774', 'MVI_40775']
------------------------------------------------------------
rainy: 9 sequences
['MVI_40863', 'MVI_40864', 'MVI_40891', 'MVI_40892', 'MVI_40901', 'MVI_40902', 'MVI_40903', 'MVI_40904', 'MVI_40905']
------------------------------------------------------------


In [37]:
wanted_conditions = ["sunny", "cloudy", "rainy", "night"]

condition_sequences = {
    cond: condition_sequences.get(cond, [])
    for cond in wanted_conditions
}

for condition, seqs in condition_sequences.items():
    print(f"{condition}: {len(seqs)} sequences")
    print(seqs[:10])

sunny: 8 sequences
['MVI_39031', 'MVI_39051', 'MVI_39211', 'MVI_39271', 'MVI_39311', 'MVI_39361', 'MVI_39371', 'MVI_39401']
cloudy: 11 sequences
['MVI_39501', 'MVI_39511', 'MVI_40701', 'MVI_40711', 'MVI_40712', 'MVI_40714', 'MVI_40851', 'MVI_40852', 'MVI_40853', 'MVI_40854']
rainy: 9 sequences
['MVI_40863', 'MVI_40864', 'MVI_40891', 'MVI_40892', 'MVI_40901', 'MVI_40902', 'MVI_40903', 'MVI_40904', 'MVI_40905']
night: 12 sequences
['MVI_40742', 'MVI_40743', 'MVI_40761', 'MVI_40762', 'MVI_40763', 'MVI_40771', 'MVI_40772', 'MVI_40773', 'MVI_40774', 'MVI_40775']


In [38]:
# Séquences réellement utilisées dans eval_stream
eval_sequences = sorted(set([sid for sid, _ in eval_stream]))

condition_sequences_eval = {}

for condition, seqs in condition_sequences.items():
    seqs_eval = [s for s in seqs if s in eval_sequences]
    condition_sequences_eval[condition] = seqs_eval

print("Condition sequences used in evaluation:")
for condition, seqs in condition_sequences_eval.items():
    print(f"{condition}: {len(seqs)} sequences")
    print(seqs)
    print("-" * 60)

Condition sequences used in evaluation:
sunny: 0 sequences
[]
------------------------------------------------------------
cloudy: 11 sequences
['MVI_39501', 'MVI_39511', 'MVI_40701', 'MVI_40711', 'MVI_40712', 'MVI_40714', 'MVI_40851', 'MVI_40852', 'MVI_40853', 'MVI_40854', 'MVI_40855']
------------------------------------------------------------
rainy: 9 sequences
['MVI_40863', 'MVI_40864', 'MVI_40891', 'MVI_40892', 'MVI_40901', 'MVI_40902', 'MVI_40903', 'MVI_40904', 'MVI_40905']
------------------------------------------------------------
night: 12 sequences
['MVI_40742', 'MVI_40743', 'MVI_40761', 'MVI_40762', 'MVI_40763', 'MVI_40771', 'MVI_40772', 'MVI_40773', 'MVI_40774', 'MVI_40775', 'MVI_40792', 'MVI_40793']
------------------------------------------------------------


In [39]:
condition_sequences = condition_sequences_eval

In [40]:
def make_cache_key(sid, img_path):
    """
    Format utilisé dans pred_cache, delta_cache, eval_gt, eval_latency, etc.
    Exemple:
    MVI_39031/MVI_39031_img00001
    """
    return f"{sid}/{Path(img_path).stem}"


def filter_stream_by_sequences(stream, sequence_list):
    """
    Garde uniquement les frames appartenant aux séquences demandées.
    """
    seq_set = set(sequence_list)
    return [(sid, img_path) for sid, img_path in stream if sid in seq_set]


def filter_dict_by_stream(data_dict, stream):
    """
    Filtre un dictionnaire en gardant seulement les clés présentes dans stream.
    """
    keys = set(make_cache_key(sid, img_path) for sid, img_path in stream)
    return {k: v for k, v in data_dict.items() if k in keys}

In [48]:
def build_fixed_stride_policy(
    stream,
    pred_cache,
    stride=3
):
    """
    Fixed-stride inference policy.

    Important:
    predictions are copied EXACTLY from pred_cache.
    No column is removed or reformatted.
    """

    preds_by_key = {}
    infer_flags = {}

    previous_sid = None
    frame_idx = 0
    last_preds = None

    for sid, img_path in stream:

        key = make_key(sid, img_path)

        # ====================================================
        # New video sequence
        # ====================================================

        if sid != previous_sid:

            frame_idx = 0
            last_preds = None

        # ====================================================
        # Detector execution
        # ====================================================

        run_detector = (
            frame_idx % stride == 0
        )

        if run_detector:

            last_preds = np.asarray(
                pred_cache[key]
            ).copy()

            infer_flags[key] = 1

        else:

            infer_flags[key] = 0

        # ====================================================
        # Direct last-output reuse
        # ====================================================

        if last_preds is None:
            raise RuntimeError(
                f"No previous prediction available for {key}"
            )

        preds_by_key[key] = last_preds.copy()

        frame_idx += 1
        previous_sid = sid

    return preds_by_key, infer_flags

In [50]:
# ============================================================
# BUILD EVALUATION-ONLY PREDICTION CACHE
# ============================================================

eval_keys = [
    make_key(sid, img_path)
    for sid, img_path in eval_stream
]

pred_eval = {
    key: pred_cache[key]
    for key in eval_keys
    if key in pred_cache
}

print("Evaluation frames :", len(eval_stream))
print("Predictions found :", len(pred_eval))

Evaluation frames : 44204
Predictions found : 44204


In [51]:
stride3_preds, stride3_flags = build_fixed_stride_policy(
    stream=eval_stream,
    pred_cache=pred_eval,
    stride=3
)

In [52]:
condition_rows = []

for condition, seq_list in condition_sequences.items():

    if len(seq_list) == 0:
        print(f"[SKIP] {condition}: no sequences in evaluation split.")
        continue

    cond_stream = filter_stream_by_sequences(eval_stream, seq_list)

    if len(cond_stream) == 0:
        print(f"[SKIP] {condition}: no frames found in eval_stream.")
        continue

    print(f"\nCondition: {condition}")
    print(f"Sequences: {len(seq_list)}")
    print(f"Frames: {len(cond_stream)}")

    # Filtrer GT et latences
    cond_gt = filter_dict_by_stream(eval_gt, cond_stream)
    cond_latency = filter_dict_by_stream(eval_latency, cond_stream)

    # Stride k3
    str_preds = filter_dict_by_stream(stride3_preds, cond_stream)
    str_flags = filter_dict_by_stream(stride3_flags, cond_stream)



    policies = [
        ("Fixed Stride K=3", str_preds, str_flags),
    ]

    for policy_name, preds, flags in policies:
        summary = summarize_policy(
            "Fixed Stride K=3",
            cond_stream,
            preds,
            flags,
            cond_gt,
            cond_latency,
            trigger_latency_per_frame=0.0
        )

        summary["condition"] = condition
        summary["num_sequences"] = len(seq_list)
        summary["num_frames_condition"] = len(cond_stream)

        condition_rows.append(summary)

condition_df = pd.DataFrame(condition_rows)

display(condition_df)

condition_df.to_csv(
    RESULTS_DIR / "condition_wise_policy_results.csv",
    index=False
)

print("Saved:", RESULTS_DIR / "condition_wise_policy_results.csv")

[SKIP] sunny: no sequences in evaluation split.

Condition: cloudy
Sequences: 11
Frames: 12825


/tmp/ipykernel_58/4064674785.py:27: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  ap = np.trapz(np.interp(x, mrec, mpre), x)



Condition: rainy
Sequences: 9
Frames: 12900

Condition: night
Sequences: 12
Frames: 18479


,policy,mAP50,mAP50_95,precision,recall,f1,count_MAE,temporal_stability,total_frames,yolo_frames,skipped_frames,frames_processed_ratio,frame_reduction,estimated_total_time_s,estimated_yolo_time_s,mean_latency_per_frame_ms,effective_fps,condition,num_sequences,num_frames_condition
0,Fixed Stride K=3,0.901208,0.667245,0.751362,0.898110,0.818208,3.742456,0.982656,12825,4279,8546,0.333645,0.666355,76.632322,76.632322,5.975230,167.357582,cloudy,11,12825
1,Fixed Stride K=3,0.755379,0.565330,0.732354,0.730767,0.731560,4.759690,0.977754,12900,4302,8598,0.333488,0.666512,73.606549,73.606549,5.705934,175.256145,rainy,9,12900
2,Fixed Stride K=3,0.887567,0.650047,0.790663,0.853971,0.821098,1.876779,0.973985,18479,6164,12315,0.333568,0.666432,107.268238,107.268238,5.804872,172.269073,night,12,18479


Saved: /kaggle/working/ua_detrac_event_adaptive_results/condition_wise_policy_results.csv
